In [ ]:
#Install ONNX packages
!pip install onnx onnxruntime-gpu 
!pip install flask

In [1]:
#create Flask Client 
import flask
import os
import requests

url_base = 'http://192.168.50.167:28023/'
response = requests.get(url_base)
response.text

'Base Test is OK'

In [2]:
from numpy import genfromtxt
import json

#prepare dataset
inputs = genfromtxt('my_inputs.csv', delimiter=',', dtype=None)
labels = genfromtxt('my_labels.csv', delimiter=',', dtype=None)
#inputs[:5],labels[:5]

#post dataset to cloud
dict = {'inputs':inputs.tolist(), 'labels':labels.tolist()}
#print(type(dict))

json_dataset = json.dumps(dict )
#print(json_dataset)

url_baseset = 'http://192.168.50.167:28023/dataset'
response = requests.post(url_baseset, json=json_dataset)

response

<Response [200]>

In [3]:
# train model using cloud
url_toTrain = 'http://192.168.50.167:28023/train'
response = requests.get(url_toTrain)
response

<Response [200]>

In [4]:
# get training log
url_log = 'http://192.168.50.167:28023/log'
response = requests.get(url_log)
response.json()

['Epoch: 0 | Loss: 1.09116',
 'Epoch: 100 | Loss: 0.79245',
 'Epoch: 200 | Loss: 0.48326',
 'Epoch: 300 | Loss: 0.34427',
 'Epoch: 400 | Loss: 0.27203',
 'Epoch: 500 | Loss: 0.22764',
 'Epoch: 600 | Loss: 0.19752',
 'Epoch: 700 | Loss: 0.17564',
 'Epoch: 800 | Loss: 0.15899',
 'Epoch: 900 | Loss: 0.14583']

In [7]:
# inference using cloud
url_log = 'http://192.168.50.167:28023/inference'
json_inputs = json.dumps({ 'input':[1,1]})
response = requests.post(url_log, json=json_inputs)
response.json()

[0.0002836318453773856, 0.9989418387413025, 0.0007745280163362622]

In [ ]:
# download trained model in ONNX
url_onnx = 'http://192.168.50.167:28023/trainedModel'
response = requests.get(url_onnx)

f = open('trainedModel.onnx','wb')
for chunk in response.iter_content(chunk_size=512):
    if chunk:
        f.write(chunk)

f.close()

In [ ]:
# not ready !
import onnx, onnxruntime

model_name = 'trainedModel.onnx'
onnx_model = onnx.load(model_name)
onnx.checker.check_model(onnx_model)

ort_session = onnxruntime.InferenceSession(model_name, providers = ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider'])

def to_numpy(tensor):
      return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()
    
    
input = [1,0]
ort_inputs = {ort_session.get_inputs()[0].name: to_numpy(input)}
ort_outs = ort_session.run(None, ort_inputs)

ort_outs
